In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS stocks.bronze")

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
df = (
    spark.table("stocks.stage.news_raw")
    # published_at is an ISO string: '2026-07-01T13:40:05Z'
    .withColumn("date",         F.to_date(F.col("published_at")))
    .withColumn("ingested_at",  F.current_timestamp())
    .drop("published_at")
)

In [ ]:
tbl = "stocks.bronze.news"
if not spark.catalog.tableExists(tbl):
    df.write.mode("overwrite").format("delta").saveAsTable(tbl)

DeltaTable.forName(spark, tbl).alias("t").merge(
    df.alias("s"), "s.uuid = t.uuid"
).whenNotMatchedInsertAll().execute()

print(f"stocks.bronze.news: {spark.table(tbl).count()} rows")
spark.table(tbl).orderBy(F.desc("date")).show(5, truncate=False)